# 02 收集编辑：在 CounterFact 的两种关系上做 ROME 编辑并保存结果

**这个 notebook 做什么：**
1. 从 P103（母语）和 P30（大洲）里，筛出原模型本来就知道答案的事实；
2. 逐条做 ROME 编辑，记录**编辑前后**的回答和概率，以及每一层句末 token 的激活差值；
3. 把结果存成文件，后面的分析和 steering 实验都从这些文件开始，不用重复做编辑；
4. 最后一部分做第一个分析：**同一关系内**的编辑差值，是否比**不同关系之间**更相似（用 CounterFact 重做实验 2b）。

**设置**：GPU T4 x2、Internet 打开。

**时间**：每条编辑大约十几到几十秒，两种关系各 200 条，预计 2–4 小时。代码每 20 条自动存一次，会话断了重新运行会从断点继续，不会从头再来。

**和之前实验的区别：**
- 数据换成 CounterFact，数量从 12 条增加到几百条；
- 差值统一用**句末 token**（PhD 学生的建议）；
- 公共函数都放在仓库里的 `steer_utils.py`，notebook 只负责调用。

## 1. 检查 GPU

In [ ]:
!nvidia-smi

## 2. 安装 EasyEdit、下载我们的仓库

**把 `REPO_URL` 改成你自己的仓库地址。** 装完以后**一定要 Run → Restart & clear cell outputs 重启**，再从第 3 步开始。

In [ ]:
REPO_URL = 'https://github.com/<your-username>/thesis-steering.git'   # <-- change this

%cd /kaggle/working
![ -d EasyEdit ] || git clone --depth 1 https://github.com/zjunlp/EasyEdit.git
!rm -rf thesis-steering && git clone --depth 1 {REPO_URL} thesis-steering
!grep -v -E "^(torch|torchvision|PyYAML|sentencepiece|av|opencv-python|numpy|scipy|scikit-learn|pandas|matplotlib|pyarrow)==" EasyEdit/requirements.txt > req.txt
!pip install -q -r req.txt && echo '=== install finished (dependency-conflict messages above can be ignored) ==='

## 3. 加载模型和数据

In [ ]:
import sys, os, time, random
sys.path.insert(0, '/kaggle/working/thesis-steering')
import torch, pandas as pd, matplotlib.pyplot as plt
import steer_utils as su

editor, model, tok, hparams = su.load_editor('/kaggle/working/EasyEdit')
data = su.load_counterfact('/kaggle/working/counterfact.json')

OUT = '/kaggle/working/results'
os.makedirs(OUT, exist_ok=True)
print('model loaded; CounterFact records:', len(data))

## 4. 实验设置

- `RELATIONS`：要用的关系；
- `N_EDITS`：每种关系最多保留多少条**编辑成功**的事实；
- `SEED`：随机种子，保证每次打乱的顺序一样，结果可以复现。

In [ ]:
RELATIONS = ['P103', 'P30']
N_EDITS = 200
SEED = 0

N_EDITS = 200 是每种关系最多收集多少条编辑成功的事实，这 200 条还要再分成两部分：

用来造向量的（你说的 training）：比如其中 70%，大约 140 条；
只用来测试的：剩下的 30%，大约 60 条。测试编辑不参与造向量，这样测出来的才是"对没见过的编辑有没有用"。
怎么分，就是之前说的三种划分（随机划分、按答案组合划分、按新答案划分），这一步放在 03 号 notebook 里做。所以真正用来造向量的，会少于 200 条。

另外，这里的 "training" 不是训练神经网络，只是"用来计算平均方向或 PCA 方向的那些编辑"。

为什么先定 200：

时间：每条编辑十几到几十秒，两种关系各 200 条，大约要 2–4 小时，正好能在 Kaggle 一次会话里跑完。
够做第一轮判断：用一百多条编辑算平均方向，随机波动已经比较小了。
可能本来就不到 200：P30 里很多事实模型可能不知道，过滤之后也许只剩一百多条。
以后要加数据很容易：把 N_EDITS 改大，比如 500，重新运行那一格，它会跳过已经做过的，接着往下做。所以我的建议是先用 200 跑出第一轮结果，如果关系内的向量看起来有希望，再把数据加到全部可用的条数，让结论更可靠。

## 5. 筛选：只保留原模型本来就知道的事实

判断标准和实验 3 一样：原模型给原答案的平均对数概率**高于**新答案。模型本来就不知道的事实，没法判断"撤销后有没有回到原答案"，所以要排除。

这一步只做前向计算，几分钟就够。

**话说如果一个事实模型不知道，不能靠加载数据集去训练吗？还是说因为当前目标是不训练所以只能选取模型已知的数据**

技术上可以：先在 CounterFact 的真实事实上微调（fine-tune）模型，让它"学会"那些原本不知道的事实。但对我们的实验来说，这样做弊大于利，而且和"不训练"的目标关系不大，主要原因是下面三个：

**1. 会改变"原模型"本身。**
我们的整个实验都是在比较"原模型"和"被编辑过的模型"。如果先微调，原模型就变成了一个"微调过的 GPT-2 XL"。而微调本身就是在大面积改权重，相当于一次很大的编辑，它留下的痕迹会和 ROME 编辑的痕迹混在一起，让结果变得很难解释。

**2. 没有必要。**
P103 有 919 条，就算模型只知道其中一半，也还有好几百条，完全够用。为了多出几十条数据去微调模型，不划算。

**3. 确实也和"不训练、不申请服务器"的目标冲突。**
微调 GPT-2 XL 需要的显存和时间，比我们现在做的多得多，Kaggle 很可能扛不住。

**为什么要只选模型已知的事实：** 原因不在于"不知道"本身有问题，而在于我们的**评价标准**。我们判断撤销成功，看的是"原答案的分数重新高于新答案"。如果原模型本来就更偏向新答案，或者两个都不偏向，那编辑前后几乎没区别，"撤销成功"就无从判断了。

**另一种思路，可以以后考虑：** 我们真正要恢复的，其实是"**原模型原来的行为**"，而不是"真实的事实"。所以也可以换一种指标：比较 steering 后的模型和原模型，输出的概率分布有多接近（比如用 KL 散度）。这样就算模型本来不知道某个事实，也能衡量"有没有回到原样"，不用过滤数据。不过这个指标不像"原答案有没有胜出"那么直观，所以现阶段先用过滤的办法，结果更好解释。

In [ ]:
known = {}
for rel in RELATIONS:
    recs = su.relation_records(data, rel)
    random.Random(SEED).shuffle(recs)
    t0 = time.time()
    known[rel] = [r for r in recs if su.score_record(model, tok, r)['prefers_true']]
    print(f'{rel}: {len(known[rel])} of {len(recs)} known '
          f'({len(known[rel]) / len(recs):.0%}), {time.time() - t0:.0f}s')

**先想一想**：P30 里大部分原答案是 Antarctica（南极洲）。你觉得 GPT-2 XL 知道的比例会比 P103 高还是低？

比我预想的高，尤其是 P30。不过要注意，这个"知道"的标准比较宽松。

这里的"知道"只表示：在原答案和新答案这两个选项之间，模型更偏向原答案。 它不等于模型自己续写时会说出原答案。比如一条 P30 的事实，原答案是 Antarctica、新答案是 Europe，模型可能只是因为主语的名字听起来像南极的东西，就更偏向 Antarctica，并不代表它真的知道这个地方在哪。

这个比例和 ROME 论文的数据对得上。论文里未编辑的 GPT-2 XL 在 CounterFact 上，"新答案概率高于原答案"的比例是 22.2%，反过来说，大约 78% 的情况偏向原答案。我们两种关系分别是 92% 和 75%，在这个范围附近。

对实验的影响：

数据很充足。 P103 有 847 条可用，P30 有 721 条，远多于 200 条。如果第一轮结果有希望，可以把 N_EDITS 调大，用上更多数据。
可以加一个更严格的子集。 在 03 号 notebook 里，除了现在的标准，再单独统计一组"模型最想说的下一个词，正好就是原答案"的事实，也就是 top-1 等于原答案的那些。在这个严格子集上，"撤销成功"的含义最清楚：模型真的变回了它原来会说的话。两组结果都报告，结论更稳。
现在这样的数据量，第一轮实验完全够了，继续等它跑完就行。

Q:我感觉应该要不先找到200条fact，直接用模型回答下看得到的是不是预期 A：不只看两个答案谁的概率高，而是看模型自己说出来的话里有没有原答案。用续写文字来判断更直观，你自己一眼就能看出模型说得对不对。

这格在做什么：

continue_text：让原模型续写 10 个 token，每一步都选概率最高的词。 says_true：原答案是否出现在续写里，不区分大小写。所以 “is the French language” 也算对，"is French" 也算对。 says_new：新答案有没有出现。正常情况下应该很少。 最后打印比例，并列出前 20 条，你可以直接看模型实际说了什么。

看结果时注意： 这个标准也会漏判一些情况，比如同义词（soccer 和 football）、模型绕了一圈还没说到答案就被 10 个 token 截断了。所以出现 False 时，最好看一眼原文，判断是模型真不知道，还是只是换了个说法

In [ ]:
# Does the original model actually say the original answer when it continues the prompt?
def continue_text(prompt, n_tokens=10):
    ids = torch.tensor([tok(prompt)['input_ids']], device=model.device)
    with torch.no_grad():
        out = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=n_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:].tolist())

check = {}
for rel in RELATIONS:
    rows_r = pd.DataFrame(torch.load(f'{OUT}/edits_{rel}.pt')['rows'])
    rows_r['text_original'] = rows_r['prompt'].map(continue_text)
    rows_r['says_true'] = [t.lower() in s.lower() for t, s in zip(rows_r['true'], rows_r['text_original'])]
    rows_r['says_new'] = [n.lower() in s.lower() for n, s in zip(rows_r['new'], rows_r['text_original'])]
    check[rel] = rows_r
    print(f'{rel}: original answer appears in the continuation for '
          f'{rows_r.says_true.sum()} of {len(rows_r)} facts ({rows_r.says_true.mean():.0%}); '
          f'new answer appears for {rows_r.says_new.sum()}')
    display(rows_r[['prompt', 'true', 'new', 'text_original', 'says_true']].head(20))

不至于不能用。这张表说明的是"续写文字"这个检查标准有问题，而不是编辑有问题。仔细看那些 `False`，大致分三类：

**1. 句子模板本身就不引导模型直接说语言（最多）**
- "John Banville, **a native** of the UK"
- "Norman Rockwell, **a native** of the Bronx"
- "Edward Burnett Tylor **spoke the language** of the people"

模型其实没说错，只是 "a native" 后面接地名、"spoke the language" 后面接 "of the people"，在英语里本来就很自然。CounterFact 的 P103 有 6 种模板，其中有几种根本不适合用续写来检查。

**2. 模型真的不知道或说错了（少数）**
- Bertrand Russell → French（应为 English）
- Abdul Kalam → Marathi（应为 Tamil）

**3. 反过来，也有"碰巧对"的**
- "Charles Maurras, speaker of the **French** National Assembly"
- "Paul Verhoeven, speaker of the **Dutch** parliament"

被算成 `True`，但模型其实把 "speaker" 理解成了"议长"，并没有在回答母语。

所以 55% 这个数字，同时低估了模型知道的程度，也有一些虚高。

## 那编辑还能用吗？

**能用，而且都要保留。** 关键在于想清楚我们要恢复的是什么：**是原模型原来的行为，而不是"真实的事实"。** 编辑把模型在原答案和新答案之间的偏好扭转了，steering 要做的是把这个偏好扭回来。从这个角度看，200 条编辑都有效，主要指标（原答案的概率重新高于新答案）用全部数据。

但你的直觉也是对的：要**给人看、讲得清楚的结果**，最好用"原模型确实会说出原答案"的那部分。所以 03 号实验里，结果分两层报告：

| 子集 | 怎么定义 | 用途 |
| --- | --- | --- |
| 全部编辑 | 两选一概率标准（现在这 200 条） | 主要指标，统计撤销率 |
| 严格子集 | 原模型续写里出现原答案，而且模板是直接问母语的那几种（"The mother tongue of X is"、"The native language of X is"、"X is a native speaker of"） | 展示文字前后变化，结论最清楚 |

**不需要重新编辑**，用这次存下的数据直接筛选就行。严格子集现在约 100 条，如果不够，以后把 `N_EDITS` 调大接着跑，P103 还有 600 多条可用。

Q:我感觉有点奇怪，你能看下别的论文里也是这么用counterfact的吗？就是不用部分原来的答案训练模型
A:我重新核对了一遍，结论不变：**没有论文先用 CounterFact 的原答案训练模型，都是直接在原版预训练模型上编辑。** 不过有两点上次没讲到，值得补充。（导师组那篇论文的全文我这边还是打不开，下面关于它的内容只来自搜索片段。）

## 结论为什么站得住

- CounterFact 的标准指标就是比较两个答案的概率：ES 统计的是编辑后 P[o*] > P[o^c] 的样本占比。
- 结果表里的基线就是一行未经任何处理的 GPT-2 XL，ES 只有 22.2。如果需要先训练模型，这一行就不会存在。
- 导师组的论文也是同样的流程：每条编辑只改一个事实，每种关系留下 100 条成功的编辑。

## 补充 1：你听到的"训练"可能来自哪里

知识编辑领域确实有"训练集"的说法，但训练的不是语言模型本身。MEND、SERAC 这类编辑方法需要先训练一个**辅助网络**（学会"怎么改模型"），所以数据集会分训练集和测试集。ROME 不需要这一步。你的题目里，"训练集"指的就是用来算向量的那部分编辑。

## 补充 2：导师组衡量"撤销"的标准，可能和我们不同

这点比上次说的更重要。从片段看，他们的撤销标准是对照**原模型的输出**，而不是对照真实答案：摘要里写的是撤销后能以 ≥80% 的准确率找回模型原来的输出，另一处说方法有助于找回原模型的输出分布。

这对我们有两个影响：

1. **"模型知不知道真实答案"就不那么关键了。** 只要 steering 后模型说的和原模型一样，就算撤销成功，哪怕原模型本来说的是错的。你昨天担心的那 45% 续写里没出现原答案的事实，在这个标准下照样能用。
2. **我们应该加上这个指标**，才能和他们的结果直接比较：steering 后模型最想说的下一个词，是否等于原模型最想说的词。数据已经有了，每条编辑都存了 `top1_before`，03 号 notebook 里加一列就行。

所以 03 号我建议报告三个指标：原答案概率是否重新高于新答案（ROME 的标准）、top-1 是否和原模型一致（导师组的标准）、以及续写文字（给人看）。

具体定义请你在论文原文的实验部分确认一下，或者直接问 PhD 学生一句："In your paper, is an edit counted as reversed when the model's prediction matches the original model's prediction?"

## 6. 逐条编辑并保存

每条事实依次：
1. 记录**编辑前**：模型续写的一段文字（8 个 token）、原答案和新答案的对数概率、模型最想说的下一个词（top-1）、每层句末的激活；
2. 做 ROME 编辑；
3. 把权重的改动分解成两根向量 `u`、`v`（ROME 的改动是秩一的，这两根就能完全还原它，比存整个矩阵小得多）；
4. 记录**编辑后**的同样几项，判断编辑是否成功（新答案的概率高于原答案）；
5. 还原权重，并检查还原后回答是否和编辑前一致；
6. 编辑成功的，保存差值（编辑前 − 编辑后，和我们的 steering 向量定义同号）和 `u`、`v`。

结果每 20 条存一次。如果会话中断，重新运行这一格（前面几格也要先跑），会自动跳过已经做过的。

In [ ]:
W = su.rewrite_weight(model, hparams)
W_orig = W.detach().clone()

def store_path(rel):
    return f'{OUT}/edits_{rel}.pt'

for rel in RELATIONS:
    store = torch.load(store_path(rel)) if os.path.exists(store_path(rel)) else dict(rows=[], diff={}, u={}, v={})
    done = {row['case_id'] for row in store['rows']}
    n_ok = sum(row['success'] for row in store['rows'])
    print(f'===== {rel}: {len(done)} already done, {n_ok} successful')
    t0 = time.time()
    for rec in known[rel]:
        if n_ok >= N_EDITS:
            break
        if rec['case_id'] in done:
            continue
        pre = su.score_record(model, tok, rec)
        text_pre = su.generate(model, tok, rec['prompt'], n_tokens=8)
        h_pre = su.hidden_states(model, tok, rec['prompt'])[:, -1]

        su.rome_edit(editor, rec)
        u, v, err = su.rank_one_factors(W.detach() - W_orig)
        post = su.score_record(model, tok, rec)
        text_post = su.generate(model, tok, rec['prompt'], n_tokens=8)
        h_post = su.hidden_states(model, tok, rec['prompt'])[:, -1]

        su.set_edit(W, W_orig)
        restored = su.score_record(model, tok, rec)['top1'] == pre['top1']

        success = post['logp_new'] > post['logp_true']
        store['rows'].append(dict(
            case_id=rec['case_id'], relation=rel, subject=rec['subject'], prompt=rec['prompt'],
            true=rec['true'], new=rec['new'],
            text_before=text_pre, text_after_edit=text_post,
            top1_before=pre['top1'], top1_after_edit=post['top1'],
            logp_true_before=pre['logp_true'], logp_new_before=pre['logp_new'],
            logp_true_after_edit=post['logp_true'], logp_new_after_edit=post['logp_new'],
            success=success, restored=restored, rank1_error=err))
        if success:
            store['diff'][rec['case_id']] = (h_pre - h_post).half()   # [49, 1600]
            store['u'][rec['case_id']], store['v'][rec['case_id']] = u, v
            n_ok += 1
        if len(store['rows']) % 20 == 0:
            torch.save(store, store_path(rel))
            rate = (time.time() - t0) / max(1, len(store['rows']) - len(done))
            print(f'{rel}: {len(store["rows"])} edited, {n_ok} successful, {rate:.1f}s per edit')
    torch.save(store, store_path(rel))
    pd.DataFrame(store['rows']).to_csv(f'{OUT}/edits_{rel}.csv', index=False)
    print(f'{rel} finished: {n_ok} successful edits saved')

## 7. 每条编辑的前后变化

这里有两张表，回答"编辑做了什么"：
- **第一张是具体的文字输出**：同一句话，编辑前和编辑后模型分别续写了什么。这是给人直接看的。
- **第二张是概率**：原答案、新答案在编辑前后的对数概率。统计成功率用的是这张，因为续写的措辞会变（比如说 soccer 而不是 football），只看文字容易误判。

**检查三件事：**
- `success`：大部分应该是 True；
- `restored`：应该全是 True，否则说明权重没还原干净，后面的结果都不可信；
- `rank1_error`：应该非常接近 0，说明用 `u`、`v` 两根向量就能完全还原这条编辑。

In [ ]:
rows = pd.concat([pd.read_csv(f'{OUT}/edits_{rel}.csv') for rel in RELATIONS])
print(rows.groupby('relation')[['success', 'restored']].mean().round(3))
print('max rank-one reconstruction error:', rows.rank1_error.max())
pd.set_option('display.max_colwidth', 50)
print('Generated text before and after the edit:')
display(rows[['relation', 'prompt', 'true', 'new', 'text_before', 'text_after_edit']].head(15))
print('Log-probabilities of the original and the new answer:')
cols = ['relation', 'subject', 'true', 'new', 'logp_true_before', 'logp_new_before',
        'logp_true_after_edit', 'logp_new_after_edit', 'success']
display(rows[cols].head(15).round(2))

## 8. 下载结果

在右侧 **Output** 面板里，把 `results/` 下的文件下载到自己电脑（`.pt` 是差值和编辑向量，`.csv` 是上面那张表）。

后面的 steering 实验也在 Kaggle 上跑，需要用到这些文件。最方便的办法是：在 Kaggle 上新建一个 **Dataset**，把这几个文件传上去，以后的 notebook 直接把它添加为输入。

---

# 分析：同一关系内的编辑，差值是否更相似？

下面几格**不需要 GPU**，可以在这里接着跑，也可以把文件下载后在自己电脑上跑。在本地运行时，把 `OUT` 改成文件所在的文件夹，并确保 `steer_utils.py` 能被导入。

## 运行之前先写下预测

1. 同一关系内（P103 内部、P30 内部）的平均余弦相似度，会比 2b 里的 0.05 高吗？高多少？
2. P30 和 P103 相比，哪个关系内部更相似？为什么？
3. 不同关系之间（P103 vs P30）会是多少？

## 9. 每一层的平均余弦相似度：关系内 vs 关系之间

In [ ]:
D = {}
for rel in RELATIONS:
    store = torch.load(f'{OUT}/edits_{rel}.pt')
    ids = sorted(store['diff'])
    D[rel] = torch.stack([store['diff'][i] for i in ids]).float()   # [n_edits, 49, 1600]
    print(rel, tuple(D[rel].shape))

n_hidden = D[RELATIONS[0]].shape[1]          # 49 for GPT-2 XL
layers = list(range(18, n_hidden))
curves = {f'within {r}': [su.mean_offdiag(su.cos_matrix(D[r][:, L])) for L in layers] for r in RELATIONS}
a, b = RELATIONS
curves[f'{a} vs {b}'] = [su.cos_matrix(D[a][:, L], D[b][:, L]).mean().item() for L in layers]

fig, ax = plt.subplots(figsize=(9, 4))
for name, ys in curves.items():
    ax.plot(layers, ys, marker='o', markersize=3, label=name)
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xlabel('hidden_states index (18 = output of edited layer 17)')
ax.set_ylabel('mean pairwise cosine similarity')
ax.set_title('Edit differences (last token): within vs across relations')
ax.legend(); plt.show()
fig.savefig(f'{OUT}/02_within_vs_across.png', dpi=150, bbox_inches='tight')

for name, ys in curves.items():
    best = max(range(len(ys)), key=lambda i: ys[i])
    print(f'{name:14} at index 18: {ys[0]:.3f}   max: {ys[best]:.3f} at index {layers[best]}')

### 讲解
- `D[rel]`：这种关系所有成功编辑的差值，叠成 `[编辑条数, 49, 1600]`。
- `su.cos_matrix(D[r][:, L])`：取第 L 个 hidden state 上所有编辑的差值，算两两之间的余弦相似度，得到一个方阵；`su.mean_offdiag` 去掉对角线后求平均，和 2b 的做法一样。
- 关系之间（`P103 vs P30`）：每条 P103 编辑和每条 P30 编辑两两比较，这里没有对角线，所以直接求平均。
- 最后打印每条曲线在第 18 个上的值，以及它的最大值出现在哪一层。

## 10. 方向由什么决定？按答案组合拆开看

在同一关系内，把两两编辑分成四类，比较各类的平均余弦相似度：
- 原答案和新答案**都相同**（例如两条都是 French → Russian）
- **只有原答案相同**（French → Russian 和 French → Dutch）
- **只有新答案相同**（French → Russian 和 English → Russian）
- **都不同**

这能告诉我们：差值方向主要由"远离原答案"决定，还是由"推向新答案"决定。

In [ ]:
def by_answer_overlap(rel, L):
    rows_r = pd.read_csv(f'{OUT}/edits_{rel}.csv')
    rows_r = rows_r[rows_r.success].set_index('case_id')
    store = torch.load(f'{OUT}/edits_{rel}.pt')
    ids = sorted(store['diff'])
    C = su.cos_matrix(torch.stack([store['diff'][i][L] for i in ids]).float())
    t = rows_r.loc[ids, 'true'].tolist()
    n = rows_r.loc[ids, 'new'].tolist()
    groups = {'same true, same new': [], 'same true only': [], 'same new only': [], 'both different': []}
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            st, sn = t[i] == t[j], n[i] == n[j]
            key = ('same true, same new' if st and sn else 'same true only' if st
                   else 'same new only' if sn else 'both different')
            groups[key].append(C[i, j].item())
    return {k: (sum(v) / len(v) if v else float('nan'), len(v)) for k, v in groups.items()}

for rel in RELATIONS:
    for L in [18, 26]:
        print(f'--- {rel}, hidden_states index {L}')
        for k, (m, cnt) in by_answer_overlap(rel, L).items():
            print(f'   {k:20} mean cos {m:6.3f}   ({cnt} pairs)')

## 11. 用自己的话写下（发给我）

1. 你的三个预测对了吗？
2. 同一关系内的相似度，比 2b 里的 0.05 高了多少？在哪一层最高？
3. 第 10 格里，四类组合的相似度怎么排序？这说明方向主要由什么决定？
4. 根据这些结果，你觉得关系内的"通用向量"有希望吗？对 P103 和 P30 分别怎么看？